# SegFormer Port Wine Stain Finetuning

This notebook trains the port wine stain segmentation model on Google Colab using CUDA GPU.

Before running this notebook, create the upload package locally from the `Back_Lumiere` project root:

```bash
zip -r port_wine_stain_training_pack.zip \
  training/SegFormer/train_port_wine_stain.py \
  training/datasets/port_wine_stain/processed \
  training/checkpoints/SegFormer/segformer_b2_melasma_colab_export \
  requirements.txt
```

Then upload `port_wine_stain_training_pack.zip` in this notebook.

## 1. Enable GPU

In Colab, go to:

`Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save`

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Upload Training Package

Upload `port_wine_stain_training_pack.zip` when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()

## 3. Unzip Project Files

In [ ]:
from pathlib import Path

zip_files = sorted(Path('/content').glob('port_wine_stain_training_pack*.zip'))
print('found zip files:', [str(p) for p in zip_files])
assert zip_files, 'Upload port_wine_stain_training_pack.zip first, then rerun this cell.'
zip_path = zip_files[-1]

!rm -rf /content/Back_Lumiere
!unzip -q "{zip_path}" -d /content/Back_Lumiere
%cd /content/Back_Lumiere
!find training/datasets/port_wine_stain/processed -maxdepth 2 -type f | wc -l

## 4. Install Training Dependencies

The smaller install command is usually enough for this training notebook.

In [ ]:
!pip install -q torch torchvision transformers safetensors pillow numpy tqdm

## 5. Verify Dataset

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
import json

dataset_dir = Path("training/datasets/port_wine_stain/processed")
print("images:", len(list((dataset_dir / "images").iterdir())))
print("masks:", len(list((dataset_dir / "masks").glob("*.png"))))
print("train:", len((dataset_dir / "splits" / "train.txt").read_text().splitlines()))
print("val:", len((dataset_dir / "splits" / "val.txt").read_text().splitlines()))
print("test:", len((dataset_dir / "splits" / "test.txt").read_text().splitlines()))

summary_path = dataset_dir / "dataset_summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text()), indent=2))

sample_name = (dataset_dir / "splits" / "train.txt").read_text().splitlines()[0]
mask = np.array(Image.open(dataset_dir / "masks" / f"{sample_name}.png"))
print("sample mask:", sample_name, "unique values:", np.unique(mask).tolist())

## 6. Smoke Test: Train 1 Epoch

Run one epoch first to confirm paths, GPU, checkpoint loading, and saving all work.

In [ ]:
!python training/SegFormer/train_port_wine_stain.py \
  --dataset-dir training/datasets/port_wine_stain/processed \
  --checkpoint training/checkpoints/SegFormer/segformer_b2_melasma_colab_export \
  --output-dir training/checkpoints/SegFormer/segformer_b2_4class_port_wine_stain_finetune_smoke_test \
  --image-size 512 \
  --batch-size 4 \
  --epochs 1 \
  --learning-rate 5e-5 \
  --device cuda

## 7. Full Training

If the smoke test succeeds, run full training. If Colab reports CUDA out of memory, change `--batch-size 4` to `--batch-size 2`.

In [ ]:
!python training/SegFormer/train_port_wine_stain.py \
  --dataset-dir training/datasets/port_wine_stain/processed \
  --checkpoint training/checkpoints/SegFormer/segformer_b2_melasma_colab_export \
  --output-dir training/checkpoints/SegFormer/segformer_b2_4class_port_wine_stain_finetune \
  --image-size 512 \
  --batch-size 4 \
  --epochs 50 \
  --learning-rate 5e-5 \
  --device cuda

## 8. Inspect Training History

In [ ]:
import json
from pathlib import Path

history_path = Path("training/checkpoints/SegFormer/segformer_b2_4class_port_wine_stain_finetune/training_history.json")
history = json.loads(history_path.read_text())
print("epochs:", len(history))
print("last:", history[-1])
print("best:", max(history, key=lambda row: row["val_port_wine_stain_iou"]))

## 9. Zip Result Checkpoint

In [ ]:
!zip -r segformer_b2_4class_port_wine_stain_finetune.zip \
  training/checkpoints/SegFormer/segformer_b2_4class_port_wine_stain_finetune

## 10. Download Result

In [ ]:
from google.colab import files

files.download("segformer_b2_4class_port_wine_stain_finetune.zip")